In [1]:
print("HELLO")

HELLO


In [3]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.5.1-cp314-cp314-win_amd64.whl (12.6 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   --------------------------------------

Cell 1: Data Preparation

In [1]:
import pandas as pd

# Update paths to point inside the "Data" folder
EMAIL_DATA_PATH = "Data/Email Analysis Dataset.csv"
CONTACTS_DATA_PATH = "Data/contacts - dimension table.csv"

print("Reading CSV files...")
emails_df = pd.read_csv(EMAIL_DATA_PATH)
contacts_df = pd.read_csv(CONTACTS_DATA_PATH)

# Drop pre-baked sentiment
if 'Sentiment' in emails_df.columns:
    emails_df = emails_df.drop(columns=['Sentiment'])
    print("✓ Successfully ignored original 'Sentiment' column.")

# Merge metadata from contacts
emails_df['From Name'] = emails_df['From Name'].str.strip()
contacts_df['Name'] = contacts_df['Name'].str.strip()

df = pd.merge(emails_df, contacts_df[['Name', 'Department color', 'Department aura']], left_on='From Name', right_on='Name', how='left')
if 'Name' in df.columns:
    df = df.drop(columns=['Name'])

df['Email topic'] = df['Email topic'].fillna('Unknown').astype(str)
df.head(3)

Reading CSV files...
✓ Successfully ignored original 'Sentiment' column.


,Email id,From Name,From seniority,From Department,To Name,To seniority,To Department,Email topic,Date,Is opened?,Device,Within work hours,Within workdays,Department color,Department aura
0,1,Leona McAree,C-level,Executive Management,Elsinore Waterland,Middle level management,Human Resources,Operational Issues,2024-03-14,opened,desktop,yes,yes,red,aura1
1,2,Leona McAree,C-level,Executive Management,Elsinore Waterland,Middle level management,Human Resources,HR related topics,2024-03-22,opened,mobile,no,yes,red,aura1
2,3,Leona McAree,C-level,Executive Management,Elsinore Waterland,Middle level management,Human Resources,Meeting Scheduling,2024-03-26,unopened,desktop,yes,yes,red,aura1


In [2]:
!pip install transformers torch tqdm

Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-5.14.0-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.13.0-cp314-cp314-win_amd64.whl.metadata (39 kB)
  Using cached tqdm-4.68.4-py3-none-any.whl.metadata (57 kB)
  Using cached huggingface_hub-1.23.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.7.10-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.27.0-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached filelock-3.29.7-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.6.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3

In [3]:
from transformers import pipeline
from tqdm.notebook import tqdm

# Load the zero-shot/pre-trained NLP transformer
# (Uses CPU execution; if using a GPU notebook instance like Kaggle/Colab, swap device=-1 to device=0)
print("Downloading/Loading model into notebook execution memory...")
sentiment_engine = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest", 
    device=-1
)

# Custom function to return predictions safely
def predict_topic(text):
    try:
        res = sentiment_engine(text)[0]
        return res['label'], res['score']
    except Exception:
        return "neutral", 0.0

# Create arrays to collect calculated metrics
predicted_labels = []
confidence_scores = []

# Loop and track with an interactive notebook progress bar
for topic in tqdm(df['Email topic'], desc="Analyzing Topics"):
    label, score = predict_topic(topic)
    predicted_labels.append(label)
    confidence_scores.append(score)

# Assign newly computed results directly to our dataframe
df['Predicted_Sentiment'] = predicted_labels
df['Confidence_Score'] = confidence_scores

print("\nProcessing complete! New columns assigned.")

Downloading/Loading model into notebook execution memory...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

C:\Users\Laurent Phil\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Laurent Phil\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Analyzing Topics:   0%|          | 0/1132 [00:00<?, ?it/s]


Processing complete! New columns assigned.


In [4]:
# 1. Display total distributions of your newly calculated labels
print("--- Newly Calculated Sentiment Overview ---")
print(df['Predicted_Sentiment'].value_counts())

# 2. View breakdown across departments
print("\n--- Custom Sentiment Distribution per Department ---")
crosstab_summary = pd.crosstab(df['From Department'], df['Predicted_Sentiment'])
display(crosstab_summary)

# 3. Save your notebook's work to a file
OUTPUT_FILE = "notebook_calculated_email_sentiment.csv"
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSuccessfully compiled results to '{OUTPUT_FILE}'")

--- Newly Calculated Sentiment Overview ---
Predicted_Sentiment
neutral     920
negative    118
positive     94
Name: count, dtype: int64

--- Custom Sentiment Distribution per Department ---


Predicted_Sentiment,negative,neutral,positive
From Department,,,
Customer Service,11,94,0
Executive Management,22,226,39
Finance and Accounting,4,53,0
Human Resources,6,33,0
Information Technology (IT),12,108,0
Legal,15,102,0
Marketing,16,124,6
Product development,15,70,48
Sales,17,110,1



Successfully compiled results to 'notebook_calculated_email_sentiment.csv'
